## 02 · Run the pipeline

Every candidate answers every golden question twice:

| Condition | What the candidate gets | Stages it measures |
| --- | --- | --- |
| `oracle` | the human-verified transcript; its TTS then speaks the human-written reference answer | 2A: reasoning on its own. 3A: speech on the same text for every candidate |
| `as_heard` | its own ASR output; its TTS then speaks its own answer | 1 (ASR), 2B (reasoning as deployed), 3B (speech of what it actually says) |

Both conditions score tool calls against the golden expected calls. Each spoken answer is re-transcribed by one fixed reference ASR (`REFERENCE_ASR`), so the round trip measures the voice, not the transcriber.

Each turn logs **one MLflow trace that holds only text**: transcripts, tool calls, the answer, and the Volume path of every audio file. Audio bytes never enter a span; `voice_evals.tracing.guard` fails the turn if one tries. Scoring happens later, offline, in `03_evaluate`.

**Illustrative data.**

In [ ]:
%pip install -q "mlflow[databricks]>=3.15" openai piper-tts
dbutils.library.restartPython()

In [ ]:
import json, os, sys

sys.path.insert(0, os.path.abspath(".."))
import mlflow
from databricks.sdk import WorkspaceClient
from voice_evals import config as C
from voice_evals.delta import replace_run
from voice_evals.pipeline import new_run_id, run_pipeline
from voice_evals.tools import default_tools

dbutils.widgets.text("run_id", "", "Run ID (blank = new)")
dbutils.widgets.text("candidates", "", "Candidates (blank = enabled in config)")
dbutils.widgets.text("reference_asr", "", "Reference ASR (blank = config)")
RUN_ID = dbutils.widgets.get("run_id").strip() or new_run_id()
NAMES = [n.strip() for n in dbutils.widgets.get("candidates").split(",") if n.strip()] or None
REFERENCE_ASR = dbutils.widgets.get("reference_asr").strip() or C.REFERENCE_ASR

experiment = C.set_experiment()
print(f"Run {RUN_ID} -> experiment {experiment.name} ({experiment.experiment_id}), reference ASR {REFERENCE_ASR}")

In [ ]:
golden = [r.asDict(recursive=True) for r in spark.table(C.TABLES["golden_utterances"])
          .where(f"dataset_version = '{C.DATASET_VERSION}'").orderBy("utterance_id").collect()]
for g in golden:
    g["expected_tool_calls"] = json.loads(g["expected_tool_calls"])

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
candidates = C.build_candidates(spark, client, NAMES)
# One fixed reference for every candidate's TTS round trip in this run, prompted to transcribe verbatim.
reference_asr = C.build_reference_asr(REFERENCE_ASR, spark, client)
for c in candidates:
    print(f"{c.name:<18} {c.models}")
print(f"{len(golden)} utterances x {len(candidates)} candidates x {len(C.CONDITIONS)} conditions = "
      f"{len(golden) * len(candidates) * len(C.CONDITIONS)} traces")

In [ ]:
rows = run_pipeline(
    candidates, golden, C.CONDITIONS, run_id=RUN_ID, tools=default_tools(),
    reference_asr=reference_asr, volume_root=C.VOLUME_ROOT, dataset_version=C.DATASET_VERSION,
)
replace_run(spark, C.TABLES["pipeline_outputs"], "pipeline_outputs", rows, RUN_ID)
errors = [r for r in rows if r.get("error")]
print(f"{len(rows)} turns, {len(errors)} errors")
for r in errors:
    print(f"  {r['candidate']} {r['utterance_id']} {r['condition']}: {r['error'][:200]}")

In [ ]:
display(spark.table(C.TABLES["pipeline_outputs"]).where(f"run_id = '{RUN_ID}'")
        .select("candidate", "condition", "utterance_id", "asr_text", "tool_calls", "agent_answer", "reasr_text", "tts_uri", "trace_id", "error")
        .orderBy("utterance_id", "candidate", "condition"))

In [ ]:
dbutils.notebook.exit(json.dumps({"run_id": RUN_ID, "turns": len(rows), "errors": len(errors)}))